In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import harmonypy as hm

In [2]:
simple_path = '/home/cavin/jt/benchmark/data/hbc/s1_filtered_cells.h5ad'
celltype_col = 'cell_type'
batch_key = "batch"
cell_emb_col = 'X_harmony'
random_seed=2026


In [3]:
adata = sc.read_h5ad(simple_path)
adata

AnnData object with n_obs × n_vars = 281780 × 313
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'control_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'batch', 'cell_type', 'n_genes'
    var: 'ensemble_id', 'type'
    obsm: 'spatial'

In [4]:
harmony_out = hm.run_harmony(adata.X.toarray(), adata.obs, batch_key,random_state=random_seed)

2026-03-12 16:22:16,649 - harmonypy - INFO - Running Harmony (PyTorch on cuda)
2026-03-12 16:22:16,649 - harmonypy - INFO -   Parameters:
2026-03-12 16:22:16,650 - harmonypy - INFO -     max_iter_harmony: 10
2026-03-12 16:22:16,650 - harmonypy - INFO -     max_iter_kmeans: 20
2026-03-12 16:22:16,650 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-03-12 16:22:16,650 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-03-12 16:22:16,651 - harmonypy - INFO -     nclust: 100
2026-03-12 16:22:16,651 - harmonypy - INFO -     block_size: 0.05
2026-03-12 16:22:16,651 - harmonypy - INFO -     lamb: [1. 1.]
2026-03-12 16:22:16,651 - harmonypy - INFO -     theta: [2. 2.]
2026-03-12 16:22:16,652 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-03-12 16:22:16,652 - harmonypy - INFO -     verbose: True
2026-03-12 16:22:16,652 - harmonypy - INFO -     random_state: 2026
2026-03-12 16:22:16,652 - harmonypy - INFO -   Data: 313 PCs × 281780 cells
2026-03-12 16:22:16,652 - harmonypy 

In [5]:
adata.obsm[cell_emb_col] = harmony_out.Z_corr

In [6]:
emb = adata.obsm[cell_emb_col]
emb.shape

(281780, 313)

In [7]:
emb_df = pd.DataFrame(
    emb, 
    index=adata.obs_names,
    columns=[f"Harmony_dim_{i}" for i in range(emb.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/batch_integrate/Harmony_human_breast_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="zstd")